# Metro Manila Transit Crowd-Forecasting — Model Training

Trains a `RandomForestRegressor` that predicts hourly **turnstile entries** for LRT/MRT stations from time, calendar, and live crowdsource features, then exports the full preprocessing + model pipeline to `transit_model.joblib`.

The custom `CyclicTransformer` is imported from `processor.py` so that `main.py` can deserialize the saved pipeline at inference time.

In [ ]:
import os

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from processor import DEFAULT_MODEL_PATH, CyclicTransformer

## 1. Load data

Expects `data/synthetic_ridership.csv` with columns:
`station_id, line, direction, hour, hour_block, day_of_week, is_weekend, is_holiday, is_payday, avg_wait_time_mins, turnstile_entries`.

In [ ]:
DATA_PATH = os.path.join("data", "synthetic_ridership.csv")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at '{DATA_PATH}'.")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows.")
df.head()

## 2. Define features and target

In [ ]:
categorical_features = ["station_id", "line", "direction"]
cyclic_features = ["hour", "hour_block", "day_of_week"]
binary_features = ["is_weekend", "is_holiday", "is_payday"]
live_feature = ["avg_wait_time_mins"]
target_column = "turnstile_entries"

feature_columns = categorical_features + cyclic_features + binary_features + live_feature

X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 3. Build the pipeline

- **Categorical** → one-hot (unknown categories ignored at inference)
- **Live wait time** → constant-fill imputation (0 when no recent reports)
- **Cyclic time** → sin/cos encoding via the shared `CyclicTransformer`
- **Binary flags** → passed through

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("impute_wait", SimpleImputer(strategy="constant", fill_value=0.0), live_feature),
        ("cyclic", CyclicTransformer(), cyclic_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=12,
                min_samples_split=5,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

## 4. Train and evaluate

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"R2:   {r2_score(y_test, y_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")

## 5. Export the model

In [ ]:
joblib.dump(pipeline, DEFAULT_MODEL_PATH)
print(f"Saved pipeline to '{DEFAULT_MODEL_PATH}'.")

## 6. Sanity check with the production predictor

In [ ]:
from processor import TransitPredictor

predictor = TransitPredictor(DEFAULT_MODEL_PATH)
sample = {
    "station_id": "MRT3-NORTHAVE",
    "line": "MRT-3",
    "direction": "NB",
    "hour": 17,
    "hour_block": "17:00",
    "day_of_week": "Mon",
    "is_weekend": 0,
    "is_holiday": 0,
    "is_payday": 0,
    "avg_wait_time_mins": 7.5,
}
print(f"Predicted turnstile entries: {predictor.predict_live(sample):.2f}")

# Missing live report -> imputed to 0.0
sample_missing = {**sample, "avg_wait_time_mins": None}
print(f"Predicted (no live report):  {predictor.predict_live(sample_missing):.2f}")